# ATLAS — USGS 3DEP terrain CNN | reproducible spatial baseline

**Scope:** 180 validated NAIP-aligned v2 DEMs, terrain-only **three-class facility footprint-size classification**. This notebook replaces the experimental cells in the uploaded `USGS_CNN.ipynb`; it does **not** change source DEMs, provenance, or teammates' scripts.

**Labels:** Small `<20,000` sqft, Medium `20,000–<100,000`, Large `≥100,000`. **Evaluation:** previously saved five 2° geographic-block folds, with an **inner grouped validation split** for early stopping. Test folds are never used for fitting normalization, model selection, or early stopping. All 180 samples receive exactly one out-of-fold prediction.

**Cautions:** Seven Small examples; reconstructed NAIP footprints not independently verified; geographic block boundaries do not guarantee large train/test separation. Terrain-only classification does not establish unknown-data-center detection. This is an exploratory research baseline, not a final generalization claim.

Run from the ATLAS repository (WSL), top to bottom. Training runs on CPU or CUDA; **S3 is storage, not GPU compute**. Use a SageMaker/EC2 training environment if needed; S3 upload is deliberately not automatic.

## 1. Imports and configuration

Import the raster and PyTorch stack and set the shared seed, fold count, and training hyperparameters. Every later cell uses these values so the five spatial folds stay comparable.

In [ ]:
from pathlib import Path
import copy, json, random, time
import numpy as np
import pandas as pd
import rasterio
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Reproducibility
SEED = 42
N_FOLDS = 5

# Training
EPOCHS = 30          
PATIENCE = 7
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
TARGET_SHAPE = (64, 96)
NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CLASS_NAMES = ["Small", "Medium", "Large"]

# ATLAS repository
PROJECT_ROOT = Path(
    "/mnt/c/Users/attho/OneDrive/Desktop/College/"
    "ACM ATLAS/ATLAS"
)

# Input datasets
DEM_DIR = PROJECT_ROOT / "data/temp/usgs_naip_aligned_v2"

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/processed/usgs_naip_alignment/"
      "usgs_naip_alignment_manifest_v2.parquet"
)

REGISTRY_PATH = (
    PROJECT_ROOT / "data/processed/im3_facility_registry.csv"
)

FOLDS_PATH = (
    PROJECT_ROOT
    / "data/processed/usgs_naip_alignment/"
      "usgs_spatial_folds_v1.csv"
)

# Model outputs
OUT_DIR = PROJECT_ROOT / "outputs/usgs_cnn_spatial_v1_30epochs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Verify paths
for name, path in {
    "DEM directory": DEM_DIR,
    "Alignment manifest": MANIFEST_PATH,
    "IM3 registry": REGISTRY_PATH,
    "Spatial folds": FOLDS_PATH,
}.items():
    print(f"{name}: {path.exists()}")

print("DEM count:", len(list(DEM_DIR.glob("*_dem.tif"))))
print("Pilot DEM exists:", (DEM_DIR / "0007805491_dem.tif").exists())

print("\nDevice:", DEVICE)
print("PyTorch:", torch.__version__)
print("Output directory:", OUT_DIR)

DEM directory: True
Alignment manifest: True
IM3 registry: True
Spatial folds: False
DEM count: 180
Pilot DEM exists: True

Device: cpu
PyTorch: 2.15.0+cu132
Output directory: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_cnn_spatial_v1_30epochs


## 2. Validate the saved 180-facility cohort and folds

Load the fold manifest, alignment table, and facility registry, then check ids, labels, and geographic groups. Training should stop here if a DEM is missing or a label disagrees with the fixed footprint thresholds.

In [20]:
folds = pd.read_csv(FOLD_PATH, dtype={'id': str, 'geo_group': str})
manifest = pd.read_parquet(MANIFEST_PATH)
manifest['id'] = manifest['id'].astype(str).str.zfill(10)
registry = pd.read_csv(REGISTRY_PATH, dtype={'id': str})
registry['id'] = registry['id'].astype(str).str.zfill(10)
folds['id'] = folds['id'].str.zfill(10)

required = {'id', 'lat', 'lon', 'size_class', 'geo_group', 'fold'}
assert required.issubset(folds.columns), required - set(folds.columns)
assert len(folds) == 180 and folds['id'].is_unique
assert set(folds['id']) == set(manifest['id'])
assert set(folds['fold']) == set(range(1, N_FOLDS + 1))
assert set(folds['size_class']) == {0, 1, 2}
assert registry['id'].is_unique
joined = folds.merge(registry[['id', 'sqft']], on='id', validate='one_to_one')
assert joined['sqft'].notna().all() and (joined['sqft'] > 0).all()
expected = np.select([joined['sqft'] < 20_000, joined['sqft'] < 100_000], [0, 1], default=2)
assert np.array_equal(joined['size_class'].to_numpy(), expected), 'Saved labels differ from fixed ATLAS thresholds'
assert set(folds['geo_group']) == set(folds['geo_group'].astype(str))
assert not folds.groupby('geo_group')['fold'].nunique().gt(1).any(), 'Geographic group crosses folds'
missing = [fid for fid in folds['id'] if not (DEM_DIR / f'{fid}_dem.tif').is_file()]
assert not missing, f'Missing DEMs: {missing[:10]}'
print('Cohort:', len(folds), '| class counts:', folds['size_class'].value_counts().sort_index().to_dict())
print('Saved fold/class table:')
display(pd.crosstab(folds['fold'], folds['size_class']))

Cohort: 180 | class counts: {0: 7, 1: 50, 2: 123}
Saved fold/class table:


size_class,0,1,2
fold,,,
1,1,10,25
2,2,10,25
3,1,10,25
4,1,10,24
5,2,10,24


## 3. Dataset: local centering, fixed padding, valid-pixel mask, train-only scale

The per-DEM mean is removed. A **single pooled RMS scale**, fitted only on the current training partition, is used for train/validation/test. Padding is zero; masks distinguish real pixels from padding. A mask-aware global average pool reduces—but does not entirely eliminate—padding shortcuts.

In [21]:
class USGSTerrainDataset(Dataset):
    def __init__(self, dataframe, dem_dir, scale=1.0, target_shape=TARGET_SHAPE):
        self.df = dataframe.reset_index(drop=True).copy()
        self.dem_dir = Path(dem_dir)
        self.scale = float(scale)
        self.target_shape = tuple(target_shape)
        if not np.isfinite(self.scale) or self.scale <= 0:
            raise ValueError('scale must be positive and finite')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        facility_id = str(row['id'])
        with rasterio.open(self.dem_dir / f'{facility_id}_dem.tif') as src:
            dem = src.read(1, masked=True)
        values = dem.compressed()
        if not len(values) or not np.isfinite(values).all():
            raise ValueError(f'Invalid DEM: {facility_id}')
        h, w = dem.shape
        H, W = self.target_shape
        if h > H or w > W:
            raise ValueError(f'{facility_id}: {h}x{w} exceeds {H}x{W}')
        mean = float(values.mean())
        centered = np.asarray((dem.filled(mean) - mean) / self.scale, dtype=np.float32)
        image = np.zeros((H, W), dtype=np.float32)
        mask = np.zeros((H, W), dtype=np.float32)
        top, left = (H - h) // 2, (W - w) // 2
        image[top:top+h, left:left+w] = centered
        mask[top:top+h, left:left+w] = (~np.ma.getmaskarray(dem)).astype(np.float32)
        return {'image': torch.from_numpy(image[None]),
                'mask': torch.from_numpy(mask[None]),
                'label': torch.tensor(int(row['size_class']), dtype=torch.long),
                'id': facility_id}


def fit_scale(train_df, dem_dir):
    # Fit on TRAIN ONLY; equal weighting by valid pixels, not by facility.
    sums, count = 0.0, 0
    for fid in train_df['id']:
        with rasterio.open(Path(dem_dir) / f'{fid}_dem.tif') as src:
            dem = src.read(1, masked=True)
        v = dem.compressed().astype(np.float64)
        if not len(v) or not np.isfinite(v).all(): raise ValueError(fid)
        sums += np.square(v - v.mean()).sum()
        count += len(v)
    scale = float(np.sqrt(sums / count))
    if not np.isfinite(scale) or scale <= 0: raise ValueError('Invalid fitted scale')
    return scale

smoke = USGSTerrainDataset(folds, DEM_DIR)
batch = next(iter(DataLoader(smoke, batch_size=8, num_workers=0)))
assert batch['image'].shape == (8, 1, *TARGET_SHAPE)
assert batch['mask'].shape == batch['image'].shape
assert torch.isfinite(batch['image']).all()
print('Batch:', tuple(batch['image'].shape), '| valid fraction:', round(batch['mask'].mean().item(), 3))

Batch: (8, 1, 64, 96) | valid fraction: 0.709


## 4. Compact terrain CNN: 64-dimensional reusable embedding

The terrain encoder uses three small convolutional blocks, mask-aware spatial averaging, and a 64-d embedding. Its classification head predicts the fixed three size classes. The mask is applied to the input and pooling; the architecture is an initial baseline, not a guarantee against edge/padding shortcuts.

In [22]:
class TerrainCNN(nn.Module):
    def __init__(self, embedding_dim=64, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
        )
        self.embedding = nn.Sequential(nn.Linear(64, embedding_dim), nn.ReLU())
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def encode(self, image, mask):
        x = self.features(image * mask)
        # Area pooling of the mask avoids hard inclusion of padded pixels.
        m = torch.nn.functional.adaptive_avg_pool2d(mask, x.shape[-2:])
        pooled = (x * m).sum(dim=(2, 3)) / m.sum(dim=(2, 3)).clamp_min(1e-6)
        return self.embedding(pooled)

    def forward(self, image, mask):
        return self.classifier(self.encode(image, mask))

model = TerrainCNN().to(DEVICE)
with torch.no_grad():
    logits = model(batch['image'].to(DEVICE), batch['mask'].to(DEVICE))
    emb = model.encode(batch['image'].to(DEVICE), batch['mask'].to(DEVICE))
assert logits.shape == (8, 3) and emb.shape == (8, 64)
print('Logits:', tuple(logits.shape), '| embedding:', tuple(emb.shape))
print('Parameters:', sum(p.numel() for p in model.parameters()))

Logits: (8, 3) | embedding: (8, 64)
Parameters: 27763


## 5. Inner validation groups and training utilities

For each held-out outer fold, choose an **inner group-disjoint validation set** from the outer training facilities. We require all three classes in the inner training and validation partitions. If this is impossible, fail rather than silently using the test fold. Hyperparameters above are fixed *before* examining outer-fold performance.

In [23]:
def make_inner_split(outer_train, seed, n_attempts=250):
    splitter = GroupShuffleSplit(n_splits=n_attempts, test_size=0.2, random_state=seed)
    best = None
    for tr_idx, va_idx in splitter.split(outer_train, groups=outer_train['geo_group']):
        tr = outer_train.iloc[tr_idx].copy()
        va = outer_train.iloc[va_idx].copy()
        if set(tr['size_class']) != {0, 1, 2} or set(va['size_class']) != {0, 1, 2}:
            continue
        # Prefer a validation set near 20% and containing >1 Small where possible.
        score = abs(len(va) / len(outer_train) - 0.2) + (0.15 if (va['size_class'] == 0).sum() < 2 else 0)
        if best is None or score < best[0]: best = (score, tr, va)
    if best is None: raise RuntimeError('Could not create class-complete, group-disjoint inner validation')
    return best[1], best[2]


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    losses, ys, preds = [], [], []
    for batch in loader:
        x = batch['image'].to(DEVICE)
        m = batch['mask'].to(DEVICE)
        y = batch['label'].to(DEVICE)
        with torch.set_grad_enabled(training):
            logits = model(x, m)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        losses.append((loss.item(), len(y)))
        ys.extend(y.cpu().tolist())
        preds.extend(logits.argmax(1).detach().cpu().tolist())
    mean_loss = sum(loss * n for loss, n in losses) / sum(n for _, n in losses)
    return {'loss': mean_loss, 'macro_f1': f1_score(ys, preds, labels=[0,1,2], average='macro', zero_division=0)}


def predict(model, loader):
    model.eval()
    records = []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['image'].to(DEVICE), batch['mask'].to(DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            for i, fid in enumerate(batch['id']):
                records.append({'id': fid, 'y_true': int(batch['label'][i]),
                                'y_pred': int(probs[i].argmax()),
                                'prob_small': float(probs[i,0]),
                                'prob_medium': float(probs[i,1]),
                                'prob_large': float(probs[i,2])})
    return pd.DataFrame(records)

# Validate the inner split on every outer fold BEFORE any training.
for fold in range(1, N_FOLDS+1):
    outer_train = folds[folds['fold'] != fold]
    inner_train, inner_val = make_inner_split(outer_train, SEED + fold)
    assert not (set(inner_train['geo_group']) & set(inner_val['geo_group']))
    print(f'Fold {fold}: inner train={len(inner_train)}, val={len(inner_val)}; '
          f'val labels={inner_val.size_class.value_counts().sort_index().to_dict()}')

Fold 1: inner train=115, val=29; val labels={0: 2, 1: 12, 2: 15}
Fold 2: inner train=114, val=29; val labels={0: 2, 1: 9, 2: 18}
Fold 3: inner train=115, val=29; val labels={0: 2, 1: 10, 2: 17}
Fold 4: inner train=116, val=29; val labels={0: 2, 1: 10, 2: 17}
Fold 5: inner train=115, val=29; val labels={0: 3, 1: 9, 2: 17}


## 6. Five-fold spatial cross-validation

**Runtime:** CPU training is supported; set `EPOCHS=3` initially to smoke-test the entire workflow, then restore the intended value and rerun for the actual experiment. Five folds train separate models. Normalization is fitted on the **inner training subset** only. Checkpoints and predictions are saved locally, never automatically pushed to S3.

In [24]:
all_predictions = []
fold_metrics = []
all_histories = []

for fold in range(1, N_FOLDS + 1):
    print(f'\n===== OUTER FOLD {fold}/{N_FOLDS} =====', flush=True)
    torch.manual_seed(SEED + fold)
    outer_train = folds[folds['fold'] != fold].copy()
    outer_test = folds[folds['fold'] == fold].copy()
    inner_train, inner_val = make_inner_split(outer_train, SEED + fold)
    assert not (set(outer_train['geo_group']) & set(outer_test['geo_group']))

    scale = fit_scale(inner_train, DEM_DIR)
    train_ds = USGSTerrainDataset(inner_train, DEM_DIR, scale=scale)
    val_ds = USGSTerrainDataset(inner_val, DEM_DIR, scale=scale)
    test_ds = USGSTerrainDataset(outer_test, DEM_DIR, scale=scale)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    counts = np.bincount(inner_train['size_class'], minlength=3)
    weights = 1 / np.sqrt(counts.astype(np.float64))
    weights = weights / weights.mean()
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    model = TerrainCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_score, best_loss, best_state, best_epoch, stale = -1., float('inf'), None, 0, 0

    for epoch in range(1, EPOCHS+1):
        tr = run_epoch(model, train_loader, criterion, optimizer)
        va = run_epoch(model, val_loader, criterion)
        all_histories.append({'fold': fold, 'epoch': epoch,
                              'train_loss': tr['loss'], 'train_macro_f1': tr['macro_f1'],
                              'val_loss': va['loss'], 'val_macro_f1': va['macro_f1']})
        improved = (va['macro_f1'] > best_score + 1e-8 or
                    (abs(va['macro_f1'] - best_score) <= 1e-8 and va['loss'] < best_loss - 1e-8))
        if improved:
            best_score, best_loss = va['macro_f1'], va['loss']
            best_state = copy.deepcopy(model.state_dict())
            best_epoch, stale = epoch, 0
        else:
            stale += 1
        if epoch == 1 or epoch % 5 == 0 or improved:
            print(f'epoch {epoch:02d} train_loss={tr["loss"]:.3f} '
                  f'val_loss={va["loss"]:.3f} val_macro_f1={va["macro_f1"]:.3f}', flush=True)
        if stale >= PATIENCE:
            print('Early stopping at', epoch)
            break

    model.load_state_dict(best_state)
    pred = predict(model, test_loader)
    pred['fold'] = fold
    all_predictions.append(pred)
    metrics = {'fold': fold, 'n_test': len(pred), 'n_train_inner': len(inner_train),
               'n_val_inner': len(inner_val), 'train_scale_m': scale,
               'best_epoch': best_epoch,
               'accuracy': accuracy_score(pred.y_true, pred.y_pred),
               'balanced_accuracy': balanced_accuracy_score(pred.y_true, pred.y_pred),
               'macro_f1': f1_score(pred.y_true, pred.y_pred, labels=[0,1,2], average='macro', zero_division=0)}
    fold_metrics.append(metrics)
    torch.save({'model_state_dict': best_state, 'fold': fold, 'train_scale_m': scale,
                'best_epoch': best_epoch, 'config': {'embedding_dim': 64, 'target_shape': TARGET_SHAPE}},
               OUT_DIR / f'terrain_cnn_fold_{fold}.pt')
    print('Held-out fold metrics:', metrics, flush=True)

pd.concat(all_predictions, ignore_index=True).to_csv(OUT_DIR / 'oof_predictions.csv', index=False)
pd.DataFrame(fold_metrics).to_csv(OUT_DIR / 'fold_metrics.csv', index=False)
pd.DataFrame(all_histories).to_csv(OUT_DIR / 'training_history.csv', index=False)
print('\nSaved training artifacts to:', OUT_DIR)


===== OUTER FOLD 1/5 =====
epoch 01 train_loss=1.078 val_loss=1.097 val_macro_f1=0.270
epoch 05 train_loss=0.888 val_loss=1.115 val_macro_f1=0.227
Early stopping at 8
Held-out fold metrics: {'fold': 1, 'n_test': 36, 'n_train_inner': 115, 'n_val_inner': 29, 'train_scale_m': 4.580105115397097, 'best_epoch': 1, 'accuracy': 0.6111111111111112, 'balanced_accuracy': 0.3133333333333333, 'macro_f1': 0.29444444444444445}

===== OUTER FOLD 2/5 =====
epoch 01 train_loss=1.042 val_loss=1.079 val_macro_f1=0.255
epoch 05 train_loss=0.887 val_loss=1.358 val_macro_f1=0.298
epoch 10 train_loss=0.805 val_loss=1.576 val_macro_f1=0.317
epoch 12 train_loss=0.824 val_loss=1.696 val_macro_f1=0.333
epoch 15 train_loss=0.788 val_loss=1.644 val_macro_f1=0.298
Early stopping at 19
Held-out fold metrics: {'fold': 2, 'n_test': 37, 'n_train_inner': 114, 'n_val_inner': 29, 'train_scale_m': 4.269237077363092, 'best_epoch': 12, 'accuracy': 0.6756756756756757, 'balanced_accuracy': 0.3333333333333333, 'macro_f1': 0.268

**Locate the finished 30-epoch output folder**

Point at the directory where the 30-epoch CNN run already saved predictions and checkpoints, and list its files. Later cells read those artifacts instead of retraining.

In [25]:
from pathlib import Path

out_dir = Path(
    "/mnt/c/Users/attho/OneDrive/Desktop/College/"
    "ACM ATLAS/ATLAS/outputs/usgs_cnn_spatial_v1_30epochs"
)

print([p.name for p in out_dir.iterdir()])

['fold_metrics.csv', 'oof_predictions.csv', 'terrain_cnn_fold_1.pt', 'terrain_cnn_fold_2.pt', 'terrain_cnn_fold_3.pt', 'terrain_cnn_fold_4.pt', 'terrain_cnn_fold_5.pt', 'training_history.csv']


**Preview the saved 30-epoch predictions**

Open the out-of-fold prediction file from the completed 30-epoch run and show its columns and first rows. This is a quick look before the pooled metrics below.

In [26]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

oof = pd.read_csv(out_dir / "oof_predictions.csv")

print("Columns:", oof.columns.tolist())
print("Rows:", len(oof))
display(oof.head())

Columns: ['id', 'y_true', 'y_pred', 'prob_small', 'prob_medium', 'prob_large', 'fold']
Rows: 180


,id,y_true,y_pred,prob_small,prob_medium,prob_large,fold
0,14930068,2,2,0.325290,0.329494,0.345216,1
1,14997588,2,2,0.322523,0.325319,0.352157,1
2,16282459,1,2,0.323914,0.335700,0.340387,1
3,31800424,1,2,0.322088,0.325023,0.352889,1
4,32560290,2,2,0.322273,0.323674,0.354053,1


## 7. Aggregate out-of-fold results and compare to majority-class baseline

All 180 predictions come from models that did not train on the corresponding facility or geographic block. Pooled OOF metrics are more interpretable than single-fold Small-class recall, but uncertainty remains substantial (only 7 Small facilities).

In [17]:
oof = pd.read_csv(OUT_DIR / 'oof_predictions.csv', dtype={'id': str})
assert len(oof) == 180 and oof['id'].is_unique
assert set(oof['id']) == set(folds['id'])
print('OOF accuracy:', round(accuracy_score(oof.y_true, oof.y_pred), 4))
print('OOF balanced accuracy:', round(balanced_accuracy_score(oof.y_true, oof.y_pred), 4))
print('OOF macro-F1:', round(f1_score(oof.y_true, oof.y_pred, labels=[0,1,2], average='macro', zero_division=0), 4))
print('\nPer-class report:')
print(classification_report(oof.y_true, oof.y_pred, labels=[0,1,2], target_names=CLASS_NAMES, zero_division=0))
print('Confusion matrix (rows=true):')
display(pd.DataFrame(confusion_matrix(oof.y_true, oof.y_pred, labels=[0,1,2]),
                     index=CLASS_NAMES, columns=CLASS_NAMES))
majority = np.full(len(oof), 2)
print('\nMajority-Large dummy:')
print({'accuracy': accuracy_score(oof.y_true, majority),
       'balanced_accuracy': balanced_accuracy_score(oof.y_true, majority),
       'macro_f1': f1_score(oof.y_true, majority, labels=[0,1,2], average='macro', zero_division=0)})
print('\nPer-fold metrics:')
display(pd.read_csv(OUT_DIR / 'fold_metrics.csv'))

OOF accuracy: 0.6278
OOF balanced accuracy: 0.3141
OOF macro-F1: 0.2769

Per-class report:
              precision    recall  f1-score   support

       Small       0.00      0.00      0.00         7
      Medium       0.14      0.04      0.06        50
       Large       0.67      0.90      0.77       123

    accuracy                           0.63       180
   macro avg       0.27      0.31      0.28       180
weighted avg       0.50      0.63      0.54       180

Confusion matrix (rows=true):


,Small,Medium,Large
Small,0,0,7
Medium,0,2,48
Large,0,12,111



Majority-Large dummy:
{'accuracy': 0.6833333333333333, 'balanced_accuracy': 0.3333333333333333, 'macro_f1': 0.2706270627062706}

Per-fold metrics:


,fold,n_test,n_train_inner,n_val_inner,train_scale_m,best_epoch,accuracy,balanced_accuracy,macro_f1
0,1,36,115,29,4.580105,1,0.611111,0.313333,0.294444
1,2,37,114,29,4.269237,1,0.675676,0.333333,0.268817
2,3,36,115,29,4.589887,2,0.583333,0.300000,0.284091
3,4,35,116,29,4.214708,1,0.600000,0.291667,0.250000
4,5,36,115,29,4.577519,1,0.666667,0.333333,0.266667


**Recompute pooled metrics from the 30-epoch file**

Reload that out-of-fold file, confirm each facility was scored once on its saved fold, and recompute accuracy, balanced accuracy, and macro-F1. The classification report, confusion matrix, and mean probabilities sit next to a majority-class baseline.

In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

oof = pd.read_csv(
    out_dir / "oof_predictions.csv",
    dtype={"id": str}
)

oof["id"] = oof["id"].str.zfill(10)

# Verify every facility was evaluated exactly once.
assert len(oof) == 180
assert oof["id"].is_unique
assert set(oof["id"]) == set(folds["id"])
assert set(oof["fold"]) == set(range(1, 6))

# Verify predictions came from the saved held-out folds.
expected_folds = folds.set_index("id")["fold"]
assert (
    oof.set_index("id")["fold"]
    .sort_index()
    .equals(expected_folds.sort_index())
)

# Verify predicted labels agree with saved probabilities.
probs = oof[
    ["prob_small", "prob_medium", "prob_large"]
].to_numpy()

assert np.isfinite(probs).all()
assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-4)
assert np.array_equal(probs.argmax(axis=1), oof["y_pred"])

y_true = oof["y_true"].to_numpy()
y_pred = oof["y_pred"].to_numpy()

majority_pred = np.full_like(y_true, 2)

results = pd.DataFrame({
    "USGS CNN": [
        accuracy_score(y_true, y_pred),
        balanced_accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro")
    ],
    "Majority baseline": [
        accuracy_score(y_true, majority_pred),
        balanced_accuracy_score(y_true, majority_pred),
        f1_score(y_true, majority_pred, average="macro")
    ]
}, index=["Accuracy", "Balanced accuracy", "Macro-F1"])

display(results.round(4))

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1, 2],
    target_names=["Small", "Medium", "Large"],
    zero_division=0
))

print("\nConfusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
    index=["Actual Small", "Actual Medium", "Actual Large"],
    columns=["Pred Small", "Pred Medium", "Pred Large"]
))

print("\nPrediction distribution:")
print(oof["y_pred"].value_counts().sort_index())

print("\nAverage predicted probabilities:")
print(
    oof[["prob_small", "prob_medium", "prob_large"]]
    .mean().round(4)
)

,USGS CNN,Majority baseline
Accuracy,0.6333,0.6833
Balanced accuracy,0.3208,0.3333
Macro-F1,0.2881,0.2706



Classification report:
              precision    recall  f1-score   support

       Small       0.00      0.00      0.00         7
      Medium       0.19      0.06      0.09        50
       Large       0.68      0.90      0.77       123

    accuracy                           0.63       180
   macro avg       0.29      0.32      0.29       180
weighted avg       0.51      0.63      0.55       180


Confusion matrix:


,Pred Small,Pred Medium,Pred Large
Actual Small,0,1,6
Actual Medium,0,3,47
Actual Large,0,12,111



Prediction distribution:
y_pred
1     16
2    164
Name: count, dtype: int64

Average predicted probabilities:
prob_small     0.1675
prob_medium    0.3459
prob_large     0.4866
dtype: float64


## 8. Save experiment metadata and inspect reusable embeddings

Write the experiment settings next to the outputs, then load the fold-1 checkpoint and extract one 64-d terrain embedding. That embedding is what a later fusion model can reuse without the classification head.

In [18]:
metadata = {
    'experiment': 'usgs_terrain_cnn_spatial_v1', 'cohort_n': len(folds),
    'class_thresholds_sqft': {'small_lt': 20000, 'medium_lt': 100000},
    'outer_folds': 5, 'spatial_groups': '2-degree latitude/longitude blocks',
    'inner_validation': 'group-disjoint, selected from outer training only',
    'normalization': 'per-tile centering + inner-training-only pooled RMS scale',
    'architecture': 'small 1-channel CNN, 64-d embedding, 3-class head',
    'epochs_max': EPOCHS, 'patience': PATIENCE, 'batch_size': BATCH_SIZE,
    'lr': LR, 'seed': SEED, 'device': str(DEVICE),
    'limitations': ['only seven Small examples', 'NAIP footprints reconstructed, not independently verified',
                    '2-degree groups do not enforce a geographic buffer',
                    'terrain-only footprint-size classification, not unknown-facility detection']
}
(OUT_DIR / 'experiment_config.json').write_text(json.dumps(metadata, indent=2))
print('Saved:', OUT_DIR / 'experiment_config.json')

# Load a saved model for inference/embedding extraction (example: fold 1).
checkpoint = torch.load(OUT_DIR / 'terrain_cnn_fold_1.pt', map_location=DEVICE, weights_only=True)
encoder = TerrainCNN().to(DEVICE)
encoder.load_state_dict(checkpoint['model_state_dict'])
encoder.eval()
example = USGSTerrainDataset(folds[folds['fold'] == 1], DEM_DIR, scale=checkpoint['train_scale_m'])[0]
with torch.no_grad():
    embedding = encoder.encode(example['image'][None].to(DEVICE), example['mask'][None].to(DEVICE))
print('Reusable terrain embedding:', tuple(embedding.shape))

Saved: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_cnn_spatial_v1/experiment_config.json
Reusable terrain embedding: (1, 64)


## 9. AWS / SageMaker notes (no automatic upload)

- **S3 stores data and outputs; it does not run training.** This notebook can run locally on CPU, or on a SageMaker notebook / managed training environment with access to the DEM files and the saved fold manifest.
- For managed SageMaker training, package the reusable dataset/model/training logic into a `.py` entrypoint, stage DEMs and fold CSV in an S3 training channel, and read them from the SageMaker-provided local input directory. **Do not assume this notebook itself is a managed training entrypoint.**
- Upload `outputs/usgs_cnn_spatial_v1/` to an agreed S3 experiment prefix **only after checking metrics, IDs, and permissions**. Keep original DEMs and provenance unchanged.
- A true multimodal comparison needs the **same 180 IDs, label definitions, and outer folds**, plus availability checks for corresponding NAIP imagery. No image fusion is performed here.

**Recommended next experiments:** (1) terrain-statistics baseline on the same folds; (2) latitude/longitude-only control; (3) terrain encoder ablation; (4) NAIP-only and late fusion, after checking NAIP match quality. If tuning hyperparameters based on these five test-fold scores, report that the evaluation became exploratory/model-selection-biased and reserve a new independent cohort for final claims.